In [ ]:
USE ROLE ROLE_MALLARD;
USE WAREHOUSE ANIMAL_TASK_WH;
USE DATABASE DB_MALLARD; 
USE DB_MALLARD.HOMEWORKS;


Q1 — Set context & confirm 16 rows in the reference table

In [ ]:
DESC TABLE STUDENT.S1.WEATHER_DATA_RAW;
SELECT COUNT(*) AS row_count FROM STUDENT.S1.WEATHER_DATA_RAW;

Q2 — Make your own copy and confirm count

In [ ]:
CREATE OR REPLACE TABLE my_weather_data_raw AS
SELECT * FROM STUDENT.S1.WEATHER_DATA_RAW;

In [ ]:
SELECT COUNT(*) AS row_count FROM my_weather_data_raw;           

In [ ]:
SELECT * FROM my_weather_data_raw LIMIT 1;  

In [ ]:
SELECT column_name, data_type
FROM INFORMATION_SCHEMA.COLUMNS
WHERE table_catalog = 'DB_MALLARD'
  AND table_schema  = 'HOMEWORKS'
  AND table_name    = 'MY_WEATHER_DATA_RAW';

Q3 — Create the typed table

In [ ]:
CREATE OR REPLACE TABLE my_weather_data (
  observation_time                 TIMESTAMP_NTZ,
  air_temp_c                       NUMBER(10,2),
  air_temp_quality_code            STRING,
  air_dew_point_c                  NUMBER(10,2),
  air_dew_point_quality_code       STRING,
  atmospheric_pressure_hpa         NUMBER(10,2),
  atmospheric_pressure_quality_code STRING,
  sky_ceiling_m                    NUMBER(10,2),
  sky_ceiling_quality_code         STRING,
  visibility_distance_m            NUMBER(10,2),
  visibility_distance_quality_code STRING,
  wind_direction_deg               NUMBER(10,0),
  wind_direction_quality_code      STRING,
  wind_speed_ms                    NUMBER(10,2),
  wind_speed_quality_code          STRING
);

In [ ]:
INSERT INTO my_weather_data
SELECT
  TO_TIMESTAMP_NTZ(W:"dt"::string)                              AS observation_time,

  W:"air"."temp"::float                                         AS air_temp_c,
  W:"air"."temp-quality-code"::string                           AS air_temp_quality_code,
  W:"air"."dew-point"::float                                    AS air_dew_point_c,
  W:"air"."dew-point-quality-code"::string                      AS air_dew_point_quality_code,

  W:"atmospheric"."pressure"::float                             AS atmospheric_pressure_hpa,
  W:"atmospheric"."pressure-quality-code"::string               AS atmospheric_pressure_quality_code,

  W:"sky"."ceiling"::float                                      AS sky_ceiling_m,
  W:"sky"."ceiling-quality-code"::string                        AS sky_ceiling_quality_code,

  W:"visibility"."distance"::float                              AS visibility_distance_m,
  W:"visibility"."distance-quality-code"::string                AS visibility_distance_quality_code,

  W:"wind"."direction-angle"::number                            AS wind_direction_deg,
  W:"wind"."direction-quality-code"::string                     AS wind_direction_quality_code,
  W:"wind"."speed-rate"::float                                  AS wind_speed_ms,
  W:"wind"."speed-quality-code"::string                         AS wind_speed_quality_code
FROM MY_WEATHER_DATA_RAW;


In [ ]:
SELECT COUNT(*) AS loaded_rows FROM my_weather_data;

In [ ]:
SELECT * FROM my_weather_data LIMIT 5;

Q5 — UDF to convert °C → °F and test

In [ ]:
CREATE OR REPLACE FUNCTION get_Farenheit_Temp(c NUMBER)
RETURNS NUMBER
LANGUAGE SQL
AS
$$
  ((c/5)*9)+32
$$;

SELECT
    100 AS CELCIUS_INPUT,
    get_Farenheit_Temp(100) AS FARENHEIT_OUTPUT;

Q6 — Add and populate temp_F

In [ ]:
ALTER TABLE my_weather_data ADD COLUMN temp_F NUMBER(10,2);

UPDATE my_weather_data
SET temp_F = get_Farenheit_Temp(air_temp_c);

SELECT COUNT(*) AS rows_with_tempF
FROM my_weather_data
WHERE temp_F IS NOT NULL;


In [ ]:
SELECT
   air_temp_c, temp_f
FROM
    my_weather_data
LIMIT 5;

Q7 — Add and populate temp_class (‘Hot’ if > 20°C else ‘Cold’)

In [ ]:
ALTER TABLE my_weather_data ADD COLUMN temp_class STRING;

UPDATE my_weather_data
SET temp_class = CASE WHEN air_temp_c > 20 THEN 'Hot' ELSE 'Cold' END;

SELECT air_temp_c, temp_F, temp_class
FROM my_weather_data
ORDER BY observation_time
LIMIT 10;


Q8 — Create hot/cold tables with same structure & ensure empty

In [ ]:
CREATE OR REPLACE TABLE my_hot_weather_data  LIKE my_weather_data;
CREATE OR REPLACE TABLE my_cold_weather_data LIKE my_weather_data;

TRUNCATE TABLE my_hot_weather_data;
TRUNCATE TABLE my_cold_weather_data;

In [ ]:
SELECT 'my_hot_weather_data' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM my_hot_weather_data
UNION ALL
SELECT 'my_cold_weather_data' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM my_cold_weather_data;

Q9 — Route rows using a single INSERT ALL

In [ ]:
INSERT ALL
  WHEN temp_class = 'Hot'  THEN INTO my_hot_weather_data
  WHEN temp_class = 'Cold' THEN INTO my_cold_weather_data
SELECT *
FROM my_weather_data;


In [ ]:
SELECT 'my_hot_weather_data' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM my_hot_weather_data
UNION ALL
SELECT 'my_cold_weather_data' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM my_cold_weather_data;